In [8]:
library('dplyr')


Attachement du package : ‘dplyr’


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union




Y a 38 variable relevante de base. Pour le calcule de l'entropy, on vas uttiliser notre model linéaire. Jeton un oeuil a notre nos 38 variable de base et se qu'il en reste apres avoir appliquer notre paipline de pre-procesing et de feature selection pour le model linéaire.

In [1]:
train_values <- read.csv("train_values.csv",stringsAsFactors = T)
test_values <- read.csv("test_values.csv",stringsAsFactors = T)
train_labels <- read.csv("train_labels.csv",stringsAsFactors = T)
submission_format <- read.csv("submission_format.csv",stringsAsFactors = T)

In [29]:
# Les 38 feature relevant de notre model sont :
feature_initial <- colnames(test_values[,-1])
feature_initial

[1] "geo_level_1_id"                        
 [2] "geo_level_2_id"                        
 [3] "geo_level_3_id"                        
 [4] "count_floors_pre_eq"                   
 [5] "age"                                   
 [6] "area_percentage"                       
 [7] "height_percentage"                     
 [8] "land_surface_condition"                
 [9] "foundation_type"                       
[10] "roof_type"                             
[11] "ground_floor_type"                     
[12] "other_floor_type"                      
[13] "position"                              
[14] "plan_configuration"                    
[15] "has_superstructure_adobe_mud"          
[16] "has_superstructure_mud_mortar_stone"   
[17] "has_superstructure_stone_flag"         
[18] "has_superstructure_cement_mortar_stone"
[19] "has_superstructure_mud_mortar_brick"   
[20] "has_superstructure_cement_mortar_brick"
[21] "has_superstructure_timber"             
[22] "has_superstructure_bamboo"             
[23] "has_superstructure_rc_non_engineered"  
[24] "has_superstructure_rc_engineered"      
[25] "has_superstructure_other"              
[26] "legal_ownership_status"                
[27] "count_families"                        
[28] "has_secondary_use"                     
[29] "has_secondary_use_agriculture"         
[30] "has_secondary_use_hotel"               
[31] "has_secondary_use_rental"              
[32] "has_secondary_use_institution"         
[33] "has_secondary_use_school"              
[34] "has_secondary_use_industry"            
[35] "has_secondary_use_health_post"         
[36] "has_secondary_use_gov_office"          
[37] "has_secondary_use_use_police"          
[38] "has_secondary_use_other"

In [30]:
length(feature_initial)

[1] 38

In [3]:
# On applique notre pipline de pre-procesing et feature selection pour notre model linéaire
dataNN <- read.csv("data_target_encoding_NN.csv",stringsAsFactors = T)
testNN <- read.csv("test_target_encoding_NN.csv",stringsAsFactors = T)
dataNN.lin <- dataNN[,-c(34,37,42,45,50,54,58,68)]
testNN.lin <- testNN[,-c(34,37,42,45,50,54,58,68)]
dataNN.lin <- dataNN.lin[,-c(13,14,19,20,21,24,25,26,27,28,29,30,31,32,35,44,46,51,52,53,54,55,56,57,58,59,60,61,62)]
testNN.lin <- testNN.lin[,-c(13,14,19,20,21,24,25,26,27,28,29,30,31,32,35,44,46,51,52,53,54,55,56,57,58,59,60,61,62)]
dataNN.lin <- dataNN.lin[,-c(12,24,28,15,31,10,32)]
testNN.lin <- testNN.lin[,-c(12,24,28,15,31,10,32)]

In [28]:
# En cour de route, on a remplacer certaine variable par d'autre, via du target encoding du dummy encoding et suprimer certain de ses variable. 
# Les variable restante sont :
feature_final <- colnames(testNN.lin)
feature_final

[1] "geo_level_1_mean_damage"               
 [2] "geo_level_1_sd_damage"                 
 [3] "geo_level_2_mean_damage"               
 [4] "geo_level_2_sd_damage"                 
 [5] "geo_level_3_mean_damage"               
 [6] "geo_level_3_sd_damage"                 
 [7] "count_floors_pre_eq"                   
 [8] "age"                                   
 [9] "area_percentage"                       
[10] "has_superstructure_adobe_mud"          
[11] "has_superstructure_mud_mortar_brick"   
[12] "has_superstructure_cement_mortar_brick"
[13] "has_superstructure_bamboo"             
[14] "count_families"                        
[15] "has_secondary_use_agriculture"         
[16] "land_surface_condition_n"              
[17] "land_surface_condition_t"              
[18] "foundation_type_r"                     
[19] "foundation_type_u"                     
[20] "foundation_type_w"                     
[21] "roof_type_x"                           
[22] "ground_floor_type_f"                   
[23] "ground_floor_type_v"                   
[24] "other_floor_type_j"                    
[25] "other_floor_type_x"                    
[26] "position_t"

In [25]:
length(feature_final)

[1] 26

On y retrouve 26 variable mais certaine de ses variable découle de la même variable a l'origine comme par exemple 'geo_level_1_mean_damage' et 'geo_level_1_sd_damage'.

Si on prend en compte les variable que l'on a tranformer pour en crée plusieur, on y retrouve 18 de nos variable originel. De sorte que 20 variable d'origine n'ont pas survecue a notre pre-procesing et feature selection. 

Pour tous les variable n'ayent pas survecue, $H(y|X^{-i}) - H(y|X) = 0$. En effet, puisque notre predicteur ne voie jamais ses variable qu'on la luit retir ou pas ne change pas la prediction et donc la probabiliter de y. Il nous sufie donc de ne calcule l'entropie $H(y|X^{-i})$ que des 18 autre variable. 

On commence par calculer l'entropy de $H(y|X)$. Pour celà on commence par entrainer notre model sur les 18 feature, comme fait prècedement. Pour faire une prediction sur le testing set. 

In [148]:
X.train <- as.matrix(dataNN.lin[,setdiff(colnames(dataNN.lin),c("damage_grade_X1","damage_grade_X2","damage_grade_X3"))])
X.test <- as.matrix(testNN.lin)
y.train <- as.matrix(dataNN.lin[,c("damage_grade_X1","damage_grade_X2","damage_grade_X3")])

X.train <- cbind(matrix(1, nr = nrow(X.train), nc = 1),X.train)
beta_hat <- solve(t(X.train)%*%X.train)%*%t(X.train)%*%y.train
y.test <- cbind(matrix(1, nr = nrow(X.test), nc = 1),X.test) %*% beta_hat
pred.lin <- 1:nrow(y.test)
for (i in pred.lin){ pred.lin[i] <- which.max(y.test[i,])}

In [199]:
df <- as.data.frame(pred.lin)
colnames(df) <- c('damage_grade')

df <- df %>% group_by(damage_grade) %>% 
  summarise('H' =n (),
            .groups = 'drop')

for (i in 1:3){df$H[i] <- df$H[i]/86868}
df

damage_grade,H
<int>,<dbl>
1,0.02957361
2,0.63616061
3,0.33426578


On calcule l'entropy via la formule : $$H(y|X) = - \sum_y p(y) log(p(y)) $$

In [200]:
H <- - df$H[1]*log(df$H[1]) - df$H[2]*log(df$H[2]) - df$H[3]*log(df$H[3])
H

[1] 0.7581578

On passe maintenant au calcule des entropie $H(y|X^{-i})$. Plutôt que de faire chaque calcule individuellement, il serait plus convignant d'automatiser le procesus. Un des problème qu'on rencontre ses que sertaine feature initial sont dedoublé et d'autre non. De plus les feature qui on été modifier n'ont plus le même non.   

In [201]:
for (i in 1:26){
    dataNN.lin.copy <- dataNN.lin[,-i]
   
    X.train <- as.matrix(dataNN.lin.copy[,setdiff(colnames(dataNN.lin.copy),c("damage_grade_X1","damage_grade_X2","damage_grade_X3"))])
    X.test <- as.matrix(testNN.lin.copy)
    y.train <- as.matrix(dataNN.lin.copy[,c("damage_grade_X1","damage_grade_X2","damage_grade_X3")])

    X.train <- cbind(matrix(1, nr = nrow(X.train), nc = 1),X.train)
    beta_hat <- solve(t(X.train)%*%X.train)%*%t(X.train)%*%y.train
    y.test <- cbind(matrix(1, nr = nrow(X.test), nc = 1),X.test) %*% beta_hat
    pred.lin.copy <- 1:nrow(y.test)
    for (j in pred.lin.copy){ pred.lin.copy[j] <- which.max(y.test[j,])}
    
    ddf <- as.data.frame(pred.lin.copy)
    colnames(ddf) <- c('damage_grade')

    ddf <- ddf %>% group_by(damage_grade) %>% 
    summarise( 'A' = n(),
                .groups = 'drop')

    for (k in 1:3){ddf$A[k] <- ddf$A[k]/86868}

    df <- merge(df,ddf,by=c('damage_grade','damage_grade'),all.x=T)
    names(df)[i+2] = paste("H", i, sep = "")
}
df

damage_grade,H,H1,H2,H3,H4,H5,H6,H7,H8,⋯,H17,H18,H19,H20,H21,H22,H23,H24,H25,H26
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.02957361,0.02953907,0.03356817,0.03658424,0.04242068,0.02653451,0.01909794,0.02052539,0.02232122,⋯,0.008933094,0.002256297,0.003695262,0.007229359,0.07288069,0.1859373,0.4151932,0.2687756,0.2214740,0.2309251
2,0.63616061,0.63757655,0.63906156,0.64092646,0.63365106,0.67710780,0.67977851,0.68532716,0.68327808,⋯,0.762283004,0.756458074,0.738200488,0.611537045,0.61794907,0.5719598,0.3938735,0.5054911,0.5620712,0.5526546
3,0.33426578,0.33288438,0.32737026,0.32248929,0.32392826,0.29635769,0.30112354,0.29414744,0.29440070,⋯,0.228783902,0.241285629,0.258104250,0.381233596,0.30917024,0.2421030,0.1909334,0.2257333,0.2164549,0.2164203


In [202]:
entropie <- 0:26
for (i in entropie){ entropie[i+1] <-  -df[1,2+i]*log(df[1,2+i]) - df[2,2+i]*log(df[2,2+i]) - df[3,2+i]*log(df[3,2+i])}
entropie <- entropie - entropie[1] 
entropie <- entropie[2:27]
M <- matrix(0,26,2)
M[,1] <- entropie
M[,2] <- 1:26
M <- as.data.frame(M)
colnames(M) <- c('entropie','indice')
M <- M[order(M$entropie, decreasing = TRUE),]

In [203]:
colnames(dataNN.lin)[M$indice]

[1] "ground_floor_type_v"                   
 [2] "other_floor_type_j"                    
 [3] "position_t"                            
 [4] "other_floor_type_x"                    
 [5] "ground_floor_type_f"                   
 [6] "roof_type_x"                           
 [7] "geo_level_2_sd_damage"                 
 [8] "geo_level_2_mean_damage"               
 [9] "geo_level_1_sd_damage"                 
[10] "geo_level_1_mean_damage"               
[11] "geo_level_3_mean_damage"               
[12] "age"                                   
[13] "foundation_type_w"                     
[14] "geo_level_3_sd_damage"                 
[15] "count_floors_pre_eq"                   
[16] "has_superstructure_mud_mortar_brick"   
[17] "has_superstructure_adobe_mud"          
[18] "area_percentage"                       
[19] "has_superstructure_cement_mortar_brick"
[20] "has_superstructure_bamboo"             
[21] "has_secondary_use_agriculture"         
[22] "land_surface_condition_n"              
[23] "count_families"                        
[24] "foundation_type_u"                     
[25] "land_surface_condition_t"              
[26] "foundation_type_r"

In [204]:
M$indice

[1] 23 24 26 25 22 21  4  3  2  1  5  8 20  6  7 11 10  9 12 13 15 16 14 19 17
[26] 18

In [205]:
M

,entropie,indice
,<dbl>,<dbl>
23,0.2899370771,23
24,0.2758209663,24
26,0.2392750992,26
25,0.2307841783,25
22,0.2175962633,22
21,0.0930846457,21
4,0.0301469712,4
3,0.0129344182,3
2,0.0074836225,2


In [170]:
entropie.copy <- entropie 

In [172]:
entropie.copy <- factor(entropie)

In [180]:
for (i in 1:26){levels(entropie.copy)[i] <- feature_final[i]}

In [177]:
levels(entropie.copy)[1] <- 'a'

In [182]:
entropie.copy

[1] foundation_type_r                     
 [2] has_superstructure_cement_mortar_brick
 [3] has_superstructure_adobe_mud          
 [4] age                                   
 [5] area_percentage                       
 [6] land_surface_condition_t              
 [7] has_superstructure_bamboo             
 [8] has_superstructure_mud_mortar_brick   
 [9] roof_type_x                           
[10] ground_floor_type_f                   
[11] ground_floor_type_v                   
[12] other_floor_type_j                    
[13] other_floor_type_x                    
[14] foundation_type_w                     
[15] has_secondary_use_agriculture         
[16] land_surface_condition_n              
[17] count_floors_pre_eq                   
[18] count_families                        
[19] foundation_type_u                     
[20] position_t                            
[21] geo_level_3_sd_damage                 
[22] geo_level_3_mean_damage               
[23] geo_level_1_mean_damage               
[24] geo_level_2_sd_damage                 
[25] geo_level_2_mean_damage               
[26] geo_level_1_sd_damage                 
26 Levels: geo_level_1_mean_damage ... position_t

In [178]:
levels(entropie.copy)

[1] "a"                     "-0.229391286860443"    "-0.228687956985494"   
 [4] "-0.222206157069709"    "-0.199775549551369"    "-0.0804046189564388"  
 [7] "-0.0476910344364139"   "-0.0297120446291146"   "-0.0217620775010017"  
[10] "-0.0208671824160644"   "-0.0145521927425445"   "-0.012139187207001"   
[13] "-0.0107739244017154"   "-0.00947067203043062"  "-0.00486459744424672" 
[16] "-0.00471689194695557"  "-0.00169551839333959"  "-0.000825223978270229"
[19] "0.00304347857974596"   "0.00334929775786219"   "0.00844739830214736"  
[22] "0.00855360551752637"   "0.0115025983931826"    "0.018437345499584"    
[25] "0.023874541155181"     "0.0828356298628644"

In [181]:
entropie.copy

[1] foundation_type_r                     
 [2] has_superstructure_cement_mortar_brick
 [3] has_superstructure_adobe_mud          
 [4] age                                   
 [5] area_percentage                       
 [6] land_surface_condition_t              
 [7] has_superstructure_bamboo             
 [8] has_superstructure_mud_mortar_brick   
 [9] roof_type_x                           
[10] ground_floor_type_f                   
[11] ground_floor_type_v                   
[12] other_floor_type_j                    
[13] other_floor_type_x                    
[14] foundation_type_w                     
[15] has_secondary_use_agriculture         
[16] land_surface_condition_n              
[17] count_floors_pre_eq                   
[18] count_families                        
[19] foundation_type_u                     
[20] position_t                            
[21] geo_level_3_sd_damage                 
[22] geo_level_3_mean_damage               
[23] geo_level_1_mean_damage               
[24] geo_level_2_sd_damage                 
[25] geo_level_2_mean_damage               
[26] geo_level_1_sd_damage                 
26 Levels: geo_level_1_mean_damage ... position_t